## create RAG app with Typesense

In [5]:
import typesense

In [6]:
client = typesense.Client({
	'nodes':[{
		'host':'qju0snwe1rfd47c3p-1.a1.typesense.net',
		'port': 443,
		'protocol':'https'
	}],
	'api_key':'yMCp6r5tjFsIBMu4JvTemJdnfZUI3N3Z',
	'connection_timeout_seconds':2
})

In [7]:
client

In [8]:
# books_schema={
# 	'name':'books',
# 	'fields':[
# 		{'name':'title','type':'string'},
# 		{'name':'authors','type':'string[]','facet':True},
# 		{'name':'publication_year','type':'int32','facet':True},
# 		{'name':'ratings_count','type':'int32'},
# 		{'name':'average_rating','type':'float'}
# 	],
# 	'defautl_sorting_field':'ratings_count'
# }
# print(client.collections.create(books_schema))

ObjectAlreadyExists: [Errno 409] A collection with name `books` already exists.

In [ ]:
with open('books.jsonl','r', encoding='utf-8') as jsonl_file:
	data=jsonl_file.read()
	client.collections['books'].documents.import_(data)

In [ ]:
search_parameters={
	'q':"harry potter",
	'query_by':'title,authors',
	'sort_by':'ratings_count:desc'
}
client.collections['books'].documents.search(search_parameters)

{'facet_counts': [],
 'found': 17,
 'hits': [{'document': {'authors': ['J.K. Rowling', ' Mary GrandPré'],
    'average_rating': 4.44,
    'id': '2',
    'image_url': 'https://images.gr-assets.com/books/1474154022m/3.jpg',
    'publication_year': 1997,
    'ratings_count': 4602479,
    'title': "Harry Potter and the Philosopher's Stone"},
   'highlight': {'title': {'matched_tokens': ['Harry', 'Potter'],
     'snippet': "<mark>Harry</mark> <mark>Potter</mark> and the Philosopher's Stone"}},
   'highlights': [{'field': 'title',
     'matched_tokens': ['Harry', 'Potter'],
     'snippet': "<mark>Harry</mark> <mark>Potter</mark> and the Philosopher's Stone"}],
   'text_match': 1157451471441102969,
   'text_match_info': {'best_field_score': '2211897868289',
    'best_field_weight': 15,
    'fields_matched': 1,
    'num_tokens_dropped': 0,
    'score': '1157451471441102969',
    'tokens_matched': 2,
    'typo_prefix_score': 0}},
  {'document': {'authors': ['J.K. Rowling', ' Mary GrandPré', ' R

In [ ]:
search_parameters={
	'q':"harry potter",
	'query_by':'title,authors',
	'filter_by':'publication_year:<1998',
	'sort_by':'ratings_count:desc'
}
client.collections['books'].documents.search(search_parameters)

{'facet_counts': [],
 'found': 1,
 'hits': [{'document': {'authors': ['J.K. Rowling', ' Mary GrandPré'],
    'average_rating': 4.44,
    'id': '2',
    'image_url': 'https://images.gr-assets.com/books/1474154022m/3.jpg',
    'publication_year': 1997,
    'ratings_count': 4602479,
    'title': "Harry Potter and the Philosopher's Stone"},
   'highlight': {'title': {'matched_tokens': ['Harry', 'Potter'],
     'snippet': "<mark>Harry</mark> <mark>Potter</mark> and the Philosopher's Stone"}},
   'highlights': [{'field': 'title',
     'matched_tokens': ['Harry', 'Potter'],
     'snippet': "<mark>Harry</mark> <mark>Potter</mark> and the Philosopher's Stone"}],
   'text_match': 1157451471441102969,
   'text_match_info': {'best_field_score': '2211897868289',
    'best_field_weight': 15,
    'fields_matched': 1,
    'num_tokens_dropped': 0,
    'score': '1157451471441102969',
    'tokens_matched': 2,
    'typo_prefix_score': 0}}],
 'out_of': 9979,
 'page': 1,
 'request_params': {'collection_name

In [9]:
### langchain + typesense + groq + rag

from langchain_community.document_loaders import TextLoader
from langchain_community.vectorstores import Typesense
from langchain_text_splitters import CharacterTextSplitter

from langchain_huggingface.embeddings import HuggingFaceEmbeddings
from langchain_groq import ChatGroq

In [10]:
import os
from dotenv import load_dotenv
load_dotenv()

groq_api_key = os.getenv("GROQ_API")

In [12]:
loader = TextLoader('text.txt')
docuemnts = loader.load()
text_splitter = CharacterTextSplitter(chunk_size=100, chunk_overlap=0)
docs = text_splitter.split_documents(docuemnts)
embeddings = HuggingFaceEmbeddings()

Created a chunk of size 309, which is longer than the specified 100
Created a chunk of size 535, which is longer than the specified 100
Created a chunk of size 419, which is longer than the specified 100
Created a chunk of size 542, which is longer than the specified 100
Created a chunk of size 374, which is longer than the specified 100
Created a chunk of size 487, which is longer than the specified 100
Created a chunk of size 431, which is longer than the specified 100
Created a chunk of size 520, which is longer than the specified 100
Created a chunk of size 459, which is longer than the specified 100
Created a chunk of size 405, which is longer than the specified 100
Created a chunk of size 366, which is longer than the specified 100
Created a chunk of size 497, which is longer than the specified 100
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 19431.64it/s]
MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
--

In [13]:
docsearch=Typesense.from_documents(
	docs,
	embeddings,
	typesense_client_params={
		'host':'qju0snwe1rfd47c3p-1.a1.typesense.net',
		'port': 443,
		'protocol':'https',
		'typesense_api_key':'yMCp6r5tjFsIBMu4JvTemJdnfZUI3N3Z',
		'typesense_collection_name':'lang_chain'
	},
	
)

In [14]:
query = 'What is Ai Governance'
found_docs = docsearch.similarity_search(query)
print(found_docs[0].page_content)

Governance is not an enemy of innovation. It is part of competent system design. Clear accountability,
red-team testing, audit logs, fallback procedures, and domain-specific review improve trustworthiness.
The hard part is that institutions often adopt AI faster than they improve their measurement and
oversight. That gap is where many avoidable failures come from.


In [15]:
# retriever
retriver = docsearch.as_retriever()
retriver


VectorStoreRetriever(tags=['Typesense', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.typesense.Typesense object at 0x166a69590>, search_kwargs={})

In [18]:
query = 'What is Ai'
retriver.invoke(query)[1].page_content

'Artificial intelligence is the broad field concerned with building systems that perform tasks commonly\nassociated with human intelligence. Those tasks include perception, language understanding, planning,\ndecision-making, pattern recognition, and learning. AI is not one technique. It is an umbrella over many\napproaches, from rule-based expert systems to neural networks to probabilistic reasoning. The field\nchanges its tools often, but the underlying aim remains similar: create systems that can act intelligently\nunder uncertainty.'